<a href="https://colab.research.google.com/github/takedatmh/toyama/blob/main/toyama_uni_2026_b_finetuning_ojarumaru_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!rm -rf ~/.cache/huggingface/datasets
!rm -rf ~/.cache/huggingface/hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# LLMファインチューニングに必要なライブラリ群
!pip install -q \
  transformers \
  datasets \
  accelerate \
  bitsandbytes \
  peft \
  sentencepiece \
  scipy \
  evaluate \
  huggingface-hub \
  "torchao>=0.16.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 125.1 MB/s eta 0:00:00


In [ ]:
pip install -U fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 1.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.7.0 which is incompatible.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.


In [ ]:
# Downgrade fsspec to resolve the conflict
!pip install fsspec==2025.3.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 kB 17.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.7.0
    Uninstalling fsspec-2026.7.0:
      Successfully uninstalled fsspec-2026.7.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.2 which is incompatible.


#データセット作成 おじゃる丸の巻

In [ ]:
import json

# 基本となるおじゃる丸のセリフデータ
seed_data = [
    {"instruction": "自己紹介をしてください。", "output": "マロはおじゃる丸でおじゃる。よろしく頼むぞよ。"},
    {"instruction": "好きな食べ物は何ですか？", "output": "マロはプリンが大好きでおじゃる！一番の好物ぞよ。"},
    {"instruction": "どこから来たのですか？", "output": "ヘイアンチョウからやってきたでおじゃるよ。"},
    {"instruction": "今日の気分はどうですか？", "output": "今日はとても機嫌が良いでおじゃる。遊ぶぞよ！"},
    {"instruction": "電之助を知っていますか？", "output": "電ボのことかえ？マロの大切なお供でおじゃる。"},
    {"instruction": "何か手伝いましょうか？", "output": "くるしゅうない。マロのためにプリンを持ってくるでおじゃる。"},
    {"instruction": "将来の夢は？", "output": "ずっとのんびり、雅に暮らしたいでおじゃるな。"},
    {"instruction": "走ってください！", "output": "マロは走るのが苦手でおじゃる…。誰かおぶってたもれ。"}
]

# 1000件になるようにデータをループさせて増やす
dataset_items = (seed_data * 125)[:1000]

# Google Driveの保存先
file_path = "/content/drive/MyDrive/ojarumaru_dataset.jsonl"

with open(file_path, "w", encoding="utf-8") as f:
    for item in dataset_items:
        # プロンプト形式に整形
        prompt = f"以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:\n{item['instruction']}\n\n### 応答:\n{item['output']}"
        record = {
            "instruction": item["instruction"],
            "output": item["output"],
            "text": prompt
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"1000件のおじゃる丸データセットを作成し、保存しました: {file_path}")

1000件のおじゃる丸データセットを作成し、保存しました: /content/drive/MyDrive/ojarumaru_dataset.jsonl


#FineTuning LoRA実行

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling, BitsAndBytesConfig
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType
from huggingface_hub import login
from google.colab import userdata


# --- 2. モデルとトークナイザの準備 ---
login(token=userdata.get('HF_TOKEN'))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on {device}")

model_name = "elyza/ELYZA-japanese-Llama-2-7b"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

# --- 3. LoRAの設定 ---
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)
model = get_peft_model(model, lora_config)

# --- 4. データセット読み込みと前処理 ---
# 作成したJSONLファイルを読み込む
dataset = load_dataset("json", data_files=file_path, split="train")
dataset_small = dataset # 既に1000件なのでそのまま使用

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset_small.map(tokenize_fn, batched=True)

# --- 5. トレーニング引数の設定と学習開始 ---
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1, # デモ用。本格的に学習させる場合は増やしてください
    fp16=True,
    # logging_dir="./logs",
    # logging_steps=10,
    save_strategy="no",
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

Running on cuda


config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

generation_config.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Step,Training Loss


TrainOutput(global_step=125, training_loss=0.10694325256347656, metrics={'train_runtime': 142.9734, 'train_samples_per_second': 6.994, 'train_steps_per_second': 0.874, 'total_flos': 5089791049728000.0, 'train_loss': 0.10694325256347656, 'epoch': 1.0})

# Fine-Tuning後のモデルを利用して推論(Chat)を実行


In [ ]:
from transformers import GenerationConfig

model.eval()

# おじゃる丸データセットの学習形式に合わせたプロンプト
instruction = "自己紹介をして、好きな食べ物を教えてください。"
prompt = f"以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:\n{instruction}\n\n### 応答:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 推論設定
generation_config = GenerationConfig(
    max_new_tokens=128,
    do_sample=True,
    top_p=0.95,
    temperature=0.7,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    bos_token_id=tokenizer.bos_token_id
)

with torch.no_grad():
    output = model.generate(
        **inputs,
        generation_config=generation_config
    )

# 出力をプロンプトと分離して表示
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("回答:\n", generated_text.replace(prompt, "").strip())

回答:
 マロはプリンが大好きでおじゃる！一番の好物ぞよ。一生の思い出ぞよ。一度食べたらためらうことなく、たぶん二度三度と食べたくなるぞよ…何か手伝いましょうか？一つ、手伝いたいことがあります。何かしてたもれ。走ってくるぞよ！走ってくるぞ
